# EE5180 — Seq2Seq (Sutskever et al., 2014) on a Colab GPU

Runs **the same `src/seq2seq` code** as the local M1 setup — nothing is reimplemented here.
A T4 is roughly 3–5x faster than the M1 Pro's MPS backend, so this is the path for the
full seed grid and the ensemble rows.

**Runtime → Change runtime type → T4 GPU** before running.

> Absolute BLEU from these runs is *not* comparable to the paper's Table 1 — see the README's
> scale-gap statement. What reproduces is the direction and shape of the effects.


## 1. Get the code


In [ ]:
# Option A: upload the repo folder to Drive and point at it.
from google.colab import drive
drive.mount('/content/drive')
REPO = '/content/drive/MyDrive/seq2seq-ee5180'   # <-- adjust

# Option B (if you pushed it to git):
# !git clone <your-repo-url> /content/seq2seq-ee5180
# REPO = '/content/seq2seq-ee5180'

import os, sys
os.chdir(REPO); sys.path.insert(0, os.path.join(REPO, 'src'))
print(open('README.md').readline())


## 2. Dependencies
Colab already ships torch built for CUDA — do **not** reinstall it from `requirements.txt`.


In [ ]:
!pip -q install sacrebleu==2.6.0 sacremoses==0.2.0 pyyaml tqdm
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')


## 3. Correctness gates
Same suite as locally. The MPS/CPU test skips on CUDA; everything else must pass.


In [ ]:
!python -m pytest tests/test_smoke.py -q


## 4. Work directory + data
`EE5180_WORK` keeps corpora and checkpoints off the repo. Put it on Drive so a
disconnect doesn't lose the run — training resumes from `last.pt`.


In [ ]:
os.environ['EE5180_WORK'] = '/content/drive/MyDrive/ee5180-work'
!mkdir -p $EE5180_WORK
!./scripts/get_wmt14.sh


In [ ]:
!python scripts/subsample_parallel.py \
  --src-in  $EE5180_WORK/data/wmt14/raw/europarl.en \
  --tgt-in  $EE5180_WORK/data/wmt14/raw/europarl.fr \
  --src-out $EE5180_WORK/data/wmt14/raw/europarl.sub.en \
  --tgt-out $EE5180_WORK/data/wmt14/raw/europarl.sub.fr \
  --n 1100000 --seed 1

!python scripts/prepare_data.py --out $EE5180_WORK/data/wmt14/prepared \
  --train-src $EE5180_WORK/data/wmt14/raw/nc9.en $EE5180_WORK/data/wmt14/raw/europarl.sub.en \
  --train-tgt $EE5180_WORK/data/wmt14/raw/nc9.fr $EE5180_WORK/data/wmt14/raw/europarl.sub.fr \
  --dev-src  $EE5180_WORK/data/wmt14/raw/newstest2013.en \
  --dev-tgt  $EE5180_WORK/data/wmt14/raw/newstest2013.fr \
  --test-src $EE5180_WORK/data/wmt14/raw/newstest2014.en \
  --test-tgt $EE5180_WORK/data/wmt14/raw/newstest2014.fr \
  --src-vocab-size 32000 --tgt-vocab-size 32000 --max-train-pairs 500000 --workers 2


## 5. Size the run for this GPU


In [ ]:
!python scripts/benchmark.py --corpus-pairs 500000


## 6. Train — the two reported rows
The reversal flag is the only difference between them.


In [ ]:
!PYTHONPATH=src python -m seq2seq.train --config configs/wmt14_small.yaml --reverse-source --seed 1


In [ ]:
!PYTHONPATH=src python -m seq2seq.train --config configs/wmt14_small.yaml --forward-source --seed 1


## 7. Decode, score, plot
Writes `results/wmt14_small/` — the table, figures and sample translations.


In [ ]:
!python scripts/make_results.py --name wmt14_small \
  --data-dir $EE5180_WORK/data/wmt14/prepared \
  --runs-root $EE5180_WORK/runs/wmt14 \
  --out results/wmt14_small --beams 1 2 12 --seeds 1 \
  --scale-note "0.5M pairs vs 12M; 2x512 vs 4x1000; 32k/32k vocab vs 160k/80k. Absolute BLEU is not comparable to Table 1."

print(open('results/wmt14_small/results.md').read())


## 8. Stretch: seeds 2–3 and the ensemble rows
Cheap on a GPU, and it makes the Table 1 ensemble/beam trends visible.


In [ ]:
for seed in (2, 3):
    !PYTHONPATH=src python -m seq2seq.train --config configs/wmt14_small.yaml --reverse-source --seed {seed}

!python scripts/make_results.py --name wmt14_small_ens \
  --data-dir $EE5180_WORK/data/wmt14/prepared --runs-root $EE5180_WORK/runs/wmt14 \
  --out results/wmt14_small_ens --beams 1 2 12 --seeds 1 2 3 --ensemble
